# Salesforce CRM Notes Formatter - LoRA Fine-Tuning
This notebook demonstrates the end-to-end process of fine-tuning a base LLM (e.g., Mistral-7B-Instruct) using Parameter-Efficient Fine-Tuning (LoRA) and 4-bit quantization (QLoRA). 

The goal is to teach the model to format raw, messy meeting notes into a structured format based on our CRM Best Practices (Summary, Key Pain Points, Action Items, Next Steps).

**Note for Local Execution:** This notebook is configured with memory-saving techniques (4-bit, Gradient Checkpointing). However, Mistral-7B requires around 6-8GB of VRAM just to load in 4-bit. If you have a 4GB GPU, you should run this on a cloud provider like Google Colab (Free T4 GPU).

In [1]:
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Data Preparation
We will load our `train.jsonl` dataset and format it into the standard ChatML format. Modern instruction tuning relies on explicit user and assistant roles.

In [2]:
from datasets import load_dataset

# Load the local JSONL file
dataset = load_dataset("json", data_files="train.jsonl", split="train")

# Define our system prompt
SYSTEM_PROMPT = """You are an expert Salesforce CRM Assistant. Your task is to take raw, messy meeting notes and format them strictly according to the company's best practices.
You must extract and organize the information into the following sections exactly:
Summary:
Key Pain Points:
Action Items:
Next Steps:
Date/Time of Interaction:
"""

# Function to map to ChatML format
def format_chatml(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]}
        ]
    }

# Apply formatting
chat_dataset = dataset.map(format_chatml, remove_columns=["input", "output"])

# Look at the first formatted example
print(chat_dataset[0]["messages"])

d:\Personal Project_certificate work Space(Projects )\salesforce project\salesforce-integration\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 10 examples [00:00, 425.39 examples/s]
Map: 100%|██████████| 10/10 [00:00<00:00, 304.67 examples/s]

[{'role': 'system', 'content': "You are an expert Salesforce CRM Assistant. Your task is to take raw, messy meeting notes and format them strictly according to the company's best practices.\nYou must extract and organize the information into the following sections exactly:\nSummary:\nKey Pain Points:\nAction Items:\nNext Steps:\nDate/Time of Interaction:\n"}, {'role': 'user', 'content': 'Acme Corp – CTO Sam Green: \n○ Explored current infrastructure; using legacy CRM. \nInterested in reducing manual data entry by 40%. \nIdentified budget stage and decision timeline (Q4).'}, {'role': 'assistant', 'content': 'Summary:\n- Discussed current infrastructure; using legacy CRM at Acme Corp.\n- CTO: Sam Green.\n\nKey Pain Points:\n- Interested in reducing manual data entry by 40%.\n\nAction Items:\n- Work on automating CRM processes to reduce manual input.\n- Align with budget stage and decision timeline (Q4).\n\nNext Steps:\n- Schedule follow-up meeting for Q4 discussion.\n\nDate/Time of Inter

## 2. Model & Tokenizer Initialization (QLoRA)
We use `BitsAndBytesConfig` to load the massive Mistral-7B model in 4-bit precision, drastically reducing memory requirements.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# We use Gemma-2B-Instruct. It's a powerful 2 Billion parameter model.
model_id = "google/gemma-2b-it"

# 4-bit Quantization Config (Crucial for 4GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Set padding token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False # Disable cache for training

## 3. LoRA Configuration
Instead of training all 2 billion parameters, we freeze the base model and attach small trainable "adapters" (LoRA) to specific linear layers.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# LoRA Configuration for Gemma architecture
peft_config = LoraConfig(
    r=16, # Rank of the update matrices
    lora_alpha=32, # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach adapters
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Setup SFTTrainer and Train
We use TRL's `SFTTrainer`. Notice we use `per_device_train_batch_size=1` and `gradient_checkpointing=True` to try to squeeze this into 4GB VRAM.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./salesforce-gemma-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1, # MINIMUM batch size for memory saving
    gradient_accumulation_steps=4, # Simulate batch size of 4
    gradient_checkpointing=True,   # EXTREMELY IMPORTANT for 4GB VRAM. Trades compute for memory.
    optim="paged_adamw_8bit",      # Memory efficient optimizer
    learning_rate=2e-4,
    fp16=True,                     # Use mixed precision
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    logging_steps=5,
    save_strategy="epoch",
    dataset_text_field="messages",
    max_seq_length=256             # Truncate inputs to 256 tokens to save activation memory
)

trainer = SFTTrainer(
    model=model,
    train_dataset=chat_dataset,
    peft_config=peft_config,
    args=training_args,
    tokenizer=tokenizer
)

# Start Training! 
# (This will take a while. If you get an Out Of Memory (OOM) error, you need a larger GPU or a smaller model)
# trainer.train()

## 5. Save and Test
After training, we save the LoRA weights. We can then test the model to see if it learned the correct formatting.

In [ ]:
# Save the LoRA adapter
# trainer.model.save_pretrained("salesforce_notes_adapter")
# tokenizer.save_pretrained("salesforce_notes_adapter")

# Simple inference test (using the loaded model)
def format_crm_notes(raw_notes):
    prompt = tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": raw_notes}
    ], tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
    
    # Decode only the newly generated tokens
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

# Test it
raw_input = """TechGlobal - VP of Sales John Doe:
Was complaining about the lack of pipeline visibility.
They are evaluating 3 different CRM solutions right now.
Wants me to send him the pricing sheet by Friday."""

# print(format_crm_notes(raw_input))